In [1]:
import numpy as np
import csv

In [3]:
# ===========================================
# 1. Leitura do CSV
# ===========================================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row['survived']
            if survived == '':
                continue

            y.append(int(survived))

            # Features selecionadas
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row['pclass']) if row['pclass'] != "" else 0

            # Convertendo sexo
            sex = 1 if row['sex'] == 'female' else 0

            age = float(row['age']) if row['age'] != "" else -1

            sibsp = float(row['sibsp']) if row['sibsp'] != "" else 0
            parch = float(row['parch']) if row['parch'] != "" else 0
            fare = float(row['fare']) if row['parch'] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1)


In [4]:
# ================================
# 2. Tratamento de Dados
# ================================

def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ages != -1])

    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X

def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)

def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [5]:
# =============================
# 3. Funções do Modelo
# =============================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )

    return cost


def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta
    

In [6]:
# =====================
# 4. Treinamento
# =====================

def train(X, y):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

In [7]:
# =====================
# 5. Predição
# =====================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [9]:
# ===========================
# 6. Split treino / teste
# ===========================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arrange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [11]:
# =====================
# 7. Execução
# =====================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

# Processar teste com MESMAS transformações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

y_pred = predict(X_test, beta)

acc = accuracy(y_test, y_pred)

print("\nAcurácia:", acc)

ValueError: could not convert string to float: ''